## 1. Import Requirements

# Master Shear-Wave Splitting Workflow for Axial Seamount

This notebook provides a complete, clean workflow from raw earthquake catalog and waveform data to shear-wave splitting analysis results. The workflow follows proper sequencing and includes all necessary quality control measures. 

Instead of using catalog from Wilcock and Zhang or ML DD, we use the nlloc file for all stations from Christian's results.

## Workflow Overview

1. **Data Loading & Initial Setup** - Load earthquake catalog and station metadata
2. **Extended Time Window Creation** - Create proper time windows for waveform retrieval
3. **Waveform Data Retrieval** - Download seismic data with extended windows
4. **Quality Control Filters** - P-wave rectilinearity, SNR, and incidence angle filtering
5. **Geometric Calculations** - Back-azimuth and distance calculations
6. **Shear-Wave Splitting Analysis** - Dynamic parameter estimation and SWSPy analysis
7. **Results Processing & Visualization** - Compile and visualize splitting parameters

## Key Improvements
- Extended catalog creation moved to proper early position
- Updated P-wave polarization analysis for true incidence angles
- Integrated SNR calculations with proper S-wave timing
- Clean separation of quality control steps

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import obspy
from obspy.core.utcdatetime import UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.event import read_events
import os
import sys
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add local swspy directory to path (before other imports)
swspy_local_path = os.path.abspath('../swspy')
if swspy_local_path not in sys.path:
    sys.path.insert(0, swspy_local_path)

# Import swspy from local directory
import swspy

# Add scripts directory to path for custom modules
sys.path.append('.')
from get_all_traces import get_station_traces_batch
from splitting_functions import *
from teanby_clustering import *

# Set up plotting
plt.rcParams['figure.figsize'] = (12, 8)

print("Libraries imported successfully")
print(f"ObsPy version: {obspy.__version__}")
print(f"SWSPy available: {'Yes' if 'swspy' in sys.modules else 'No'}")
print(f"SWSPy location: {swspy.__file__}")

In [ ]:
# Load Kaiwen's ML DD catalog
catalog_file = '../data/mldd_catalog_2015_2021.csv'

catalog = pd.read_csv(catalog_file)

#Load station information from Christian's data
stations_file = '../data/stations_axial.llz'
#Read llz file - reads like a text file with space delimiter
stations_df = pd.read_csv(stations_file, delim_whitespace=True, header=None, names=['Longitude (°W)', 'Latitude (°N)', 'Elevation (m)', 'Station ID'])
# Convert elevation column to m from km
stations_df['Elevation (m)'] = stations_df['Elevation (m)']*1000

print(f"Stations in catalog: {catalog['station'].value_counts()}")

In [ ]:
# Remove leading 'OO' from station names in catalog
# Remove leading 'OO' from station names in catalog
catalog['station'] = catalog['station'].str.replace('OO', '', regex=False)
print(f"Station names after removing 'OO' prefix: {catalog['station'].unique()}")

In [ ]:
# Look at AXEC3 station
axec3_catalog = catalog[catalog['station'] == 'AXEC3'].copy()

# Reset index to ensure clean indexing
axec3_catalog = axec3_catalog.reset_index(drop=True)
# Pick all events from the week around the eruption
axec3_catalog['datetime'] = pd.to_datetime(axec3_catalog['datetime'])
axec3_catalog = axec3_catalog[(axec3_catalog['datetime'] >= '2015-04-20') & (axec3_catalog['datetime'] < '2015-04-28')].copy()


print(f"Total AXEC3 events in catalog: {len(axec3_catalog)}")

print(f"\nDate range of test catalog:")
print(f"Start: {axec3_catalog['datetime'].min()}")
print(f"End: {axec3_catalog['datetime'].max()}")

display(axec3_catalog)

In [ ]:
test_catalog = axec3_catalog.copy()

print(f"\nDate range of test catalog:")
print(f"Start: {test_catalog['datetime'].min()}")
print(f"End: {test_catalog['datetime'].max()}")

In [ ]:
# Convert to UTCDateTime
test_catalog['datetime'] = test_catalog['datetime'].apply(lambda x: UTCDateTime(x))

# Format p_time and s_time as UTCDateTime of datetime + p_arrival_time and s_arrival_time, respectively
# Format p_time and s_time as UTCDateTime of datetime + p_time and s_time, respectively
test_catalog['p_time'] = test_catalog.apply(lambda row: UTCDateTime(row['datetime']) + row['p_arrival_time'], axis=1)
test_catalog['s_time'] = test_catalog.apply(lambda row: UTCDateTime(row['datetime']) + row['s_arrival_time'], axis=1)

print("Successfully converted p_time and s_time to absolute UTCDateTime")
print(f"Sample p_time: {test_catalog['p_time'].iloc[0]}")
print(f"Sample s_time: {test_catalog['s_time'].iloc[0]}")

print("Successfully converted to UTCDateTime")
print(f"Sample p_time: {test_catalog['p_time'].iloc[0]}")

In [ ]:
catalog = test_catalog.copy()

## 3. Extended Time Window Creation

This step creates extended time windows for waveform retrieval. This is critical for proper analysis and must happen early in the workflow, before any quality control that depends on waveform data.

In [ ]:
# Create extended time windows for proper waveform analysis
print("Creating extended time windows for waveform retrieval...")

# Apply extended windowing
extended_catalog = create_extended_catalog(catalog, pre_p_time=1.0, post_s_time=2.0)

print(f"Extended catalog created with {len(extended_catalog)} events")
print(f"Time windows: {extended_catalog['total_duration'].iloc[0]} seconds total")
print(f"Pre-event: {extended_catalog['pre_p_sec'].iloc[0]}s, Post-event: {extended_catalog['post_s_sec'].iloc[0]}s")
# Display sample of extended timing
print("\nSample timing windows:")
sample_cols = ['event_id', 'datetime', 'starttime', 'endtime', 'total_duration']
display(extended_catalog[sample_cols].head())

In [ ]:
display(extended_catalog)

In [ ]:
# Rename columns p_arrival_time and s_arrival_time to p_time and s_time
catalog = catalog.rename(columns={'magnitude': 'mag'})
print(f"Renamed columns: {catalog.columns.tolist()}")

## 4. Waveform Data Retrieval

This section retrieves seismic waveform data using the extended time windows. We'll load the existing trace data and organize it for processing.

In [ ]:
test_catalog = extended_catalog

In [ ]:
# Replace catalog id with index
test_catalog['id'] = test_catalog.index

In [ ]:
def get_station_traces_batch(df, filename, starttime, endtime, station_id, batch_size=250):
    """
    Fast bulk retrieval of waveform data with intelligent batched fallback.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with event information
    filename : str
        Output filename (without extension)
    starttime : str
        Column name for start time
    endtime : str
        Column name for end time
    station_id : str
        Column name for station ID
    
    Returns:
    --------
    obspy.Stream : All retrieved traces
    
    Performance:
    - Bulk success: ~10-15 seconds for 250 events
    - Batched fallback: ~30-60 seconds (250 events per batch)
    - Individual fallback: Only for failed batches
    """
    from obspy.clients.fdsn import Client
    from obspy.core.utcdatetime import UTCDateTime
    from obspy import Stream
    import time
    
    client = Client("IRIS")
    all_traces = Stream()

    # BATCHED- 250 events per batch
    batch_size = batch_size
    total_batches = (len(df) + batch_size - 1) // batch_size
    
    for batch_idx in range(total_batches):
        batch_start_idx = batch_idx * batch_size
        batch_end_idx = min(batch_start_idx + batch_size, len(df))
        batch_df = df.iloc[batch_start_idx:batch_end_idx]
        
        print(f"\n{'─'*60}")
        print(f"BATCH {batch_idx + 1}/{total_batches}")
        print(f"Events {batch_start_idx + 1} to {batch_end_idx} ({len(batch_df)} events)")
        print(f"{'─'*60}")
        
        # Build bulk request for this batch
        batch_bulk_list = []
        for _, row in batch_df.iterrows():
            t_start = UTCDateTime(row[str(starttime)]) - 0.5
            t_final = UTCDateTime(row[str(endtime)]) + 0.5
            current_station = row[str(station_id)]
            
            if current_station == 'AXEC3':
                batch_bulk_list.append(('OO', 'AXEC3', '', 'EHE', t_start, t_final))
                batch_bulk_list.append(('OO', 'AXEC3', '', 'EHN', t_start, t_final))
                batch_bulk_list.append(('OO', 'AXEC3', '', 'EHZ', t_start, t_final))
        
        # Try batch bulk request
        batch_start_time = time.time()
        batch_stream = client.get_waveforms_bulk(batch_bulk_list)
        batch_elapsed = time.time() - batch_start_time
        
        all_traces += batch_stream
        
        print(f"✓ Batch {batch_idx + 1} SUCCESS: {len(batch_stream)} traces in {batch_elapsed:.1f}s")
        print(f"  Expected: {len(batch_bulk_list)}, Retrieved: {len(batch_stream)}")
        
        if len(batch_stream) < len(batch_bulk_list):
            missing = len(batch_bulk_list) - len(batch_stream)
            print(f"  ⚠ Warning: {missing} traces missing from this batch")
            

            
            print(f"\n{'='*60}")
            print(f"BATCHED FALLBACK COMPLETE")
            print(f"{'='*60}")
            print(f"Total traces retrieved: {len(all_traces)}")
    
    # Save results
    if all_traces:
        print(f"\nSaving {len(all_traces)} traces to {filename}.mseed...")
        all_traces.write(str(filename) + ".mseed", format="MSEED")
        print(f"✓ File saved successfully")
        
        # Summary statistics
        print(f"\n{'='*60}")
        print(f"RETRIEVAL SUMMARY")
        print(f"{'='*60}")
        print(f"Total events processed: {len(df)}")
        print(f"Total traces retrieved: {len(all_traces)}")
        print(f"Expected traces (max): {len(df) * 3}")
        print(f"Success rate: {len(all_traces)/(len(df)*3)*100:.1f}%")
        
    else:
        print(f"\n⚠ WARNING: No traces retrieved!")
    
    return all_traces

In [ ]:
# Retrieve waveforms for all events in the test catalog using get_all_traces function
print("Retrieving waveforms for all events in the test catalog...")
waveforms = get_station_traces_batch(test_catalog, 'axial_mldd_april_20_28_axec3', 'starttime', 'endtime', 'station', batch_size=100)

In [ ]:
# Load waveforms from mseed file with obspy
waveforms_file = 'axial_mldd_april_20_28_axec3.mseed'
waveforms = obspy.read(waveforms_file)

In [ ]:
# Associate waveforms with events in the catalog
print("Organizing waveforms by events...")
waveform_dict = organize_stream_by_events(waveforms, test_catalog)

In [ ]:
# Rename test_catalog 'magnitude' column to 'mag'
test_catalog = test_catalog.rename(columns={'magnitude': 'mag'})
print(f"Renamed 'magnitude' column to 'mag': {test_catalog.columns.tolist()}")

In [ ]:
# Rename test_catalog depth_km to dep 
test_catalog = test_catalog.rename(columns={'depth_km': 'dep'})
print(f"Renamed 'depth_km' column to 'dep': {test_catalog.columns.tolist()}")

In [ ]:
# Rename test_catalog depth_km to dep 
test_catalog = test_catalog.rename(columns={'latitude': 'lat', 'longitude': 'lon'})

In [ ]:
# Organize waveforms by event ID
print("Organizing waveforms by event ID...")
organized_waveforms = organize_waveform_data(waveform_dict, test_catalog)

In [ ]:
# Format s_arrival_time and p_arrival_time as difference between arrival times and origin time
print("Formatting s_arrival_time and p_arrival_time as differences from origin time...")
for eid in organized_waveforms.keys():
    organized_waveforms[eid]['s_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 's_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))
    organized_waveforms[eid]['p_arrival_time'] = (UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'p_time'].values[0]) - 
                                                  UTCDateTime(test_catalog.loc[test_catalog['id'] == eid, 'datetime'].values[0]))

In [ ]:
# For all traces in organized_waveforms, taper and filter in-place
print("Tapering and filtering all traces in organized_waveforms...")
events_to_remove = []
try:
    for eid in organized_waveforms.keys():
        for tr in organized_waveforms[eid]['traces']:
            tr.detrend("linear") # to avoid weird start and end amplitudes
            tr.taper(max_percentage=0.05, type='hann')
            tr.filter('bandpass', freqmin=5.0, freqmax=40.0)
except Exception as e:
    print(f"Error during waveform processing: {e}")
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
    print(f"Events with issues: {events_to_remove}")

print("Waveform retrieval and organization complete.")

In [ ]:
# Find streams that are NoneType and remove from organized_waveforms
print("Checking for NoneType streams in organized_waveforms...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if type(organized_waveforms[eid]['traces']) == type(None):
        events_to_remove.append(eid)
if events_to_remove:
    print(f"Removing {len(events_to_remove)} events with NoneType streams: {events_to_remove}")
    for eid in events_to_remove:
        del organized_waveforms[eid]
else:
    print("No NoneType streams found in organized_waveforms.")

In [ ]:
# Remove duplicate traces from organized_waveforms
print("Checking for and removing duplicate traces in organized_waveforms...")

for eid in organized_waveforms.keys():
    # Get the stream for this event
    st = organized_waveforms[eid]['traces']
    
    # Check if there are duplicates
    try:
        if type(st) == type(None):
            print(f"Event {eid}: No traces found (NoneType)")
            continue

        else:
            if len(st) > 3:
                print(f"Event {eid}: Found {len(st)} traces (expected 3)")
                
                # Create a new stream with unique traces based on channel code
                unique_traces = {}
                for tr in st:
                    channel = tr.stats.channel
                    # Keep the first occurrence of each channel
                    if channel not in unique_traces:
                        unique_traces[channel] = tr
                
                # Replace the stream with deduplicated traces
                organized_waveforms[eid]['traces'] = obspy.Stream(traces=list(unique_traces.values()))
                print(f"  Reduced to {len(organized_waveforms[eid]['traces'])} unique traces")
    except Exception as e:
        print(f"Error processing event {eid}: {e}")
        organized_waveforms[eid]['traces'] = st[:3]  # Fallback to first 3 traces if error occurs

# Verify the results
print("\nVerification of trace counts after deduplication:")
trace_counts = {}
for eid in organized_waveforms.keys():
    count = len(organized_waveforms[eid]['traces'])
    trace_counts[count] = trace_counts.get(count, 0) + 1

print(f"Events with 3 traces: {trace_counts.get(3, 0)}")
if any(k != 3 for k in trace_counts.keys()):
    print("Events with unexpected trace counts:")
    for count, num_events in trace_counts.items():
        if count != 3:
            print(f"  {num_events} events with {count} traces")
else:
    print("All events have exactly 3 traces (E, N, Z)")

In [ ]:
# Remove events that do not have exactly 3 traces
print("\nRemoving events that do not have exactly 3 traces...")
events_to_remove = []
for eid in organized_waveforms.keys():
    if len(organized_waveforms[eid]['traces']) != 3:
        events_to_remove.append(eid)

    # also remove events with any trace that has zero length (indicating a retrieval issue) or empty traces
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

for eid in events_to_remove:
    del organized_waveforms[eid]

print(f"Remaining events for further processing: ", {len(organized_waveforms)})

## 5. Quality Control Pipeline

This section implements comprehensive quality control measures including P-wave rectilinearity analysis, signal-to-noise ratio calculations, and incidence angle filtering.

In [ ]:
# Define quality control thresholds
QC_THRESHOLDS = {
    'min_snr': 2.0,           # Minimum S-wave signal-to-noise ratio
    'min_rectilinearity': 0.7, # Minimum P-wave rectilinearity
    'max_incidence': 30.0,     # Maximum incidence angle (degrees)
}

print("Quality control functions loaded successfully")
print(f"QC Thresholds: {QC_THRESHOLDS}")

In [ ]:
# Check that all traces for same event have same length, and remove events that do not meet this criterion
print("Checking that all traces for the same event have the same length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    trace_lengths = [tr.stats.npts for tr in organized_waveforms[eid]['traces']]
    if len(set(trace_lengths)) != 1:
        print(f"Event {eid} has traces of different lengths: {trace_lengths}, marking for removal")
        events_to_remove.append(eid)

In [ ]:
# Check if any traces are length zero, and if so mark those events for removal
print("Checking for traces with zero length...")
events_to_remove = []
for eid in organized_waveforms.keys():
    for tr in organized_waveforms[eid]['traces']:
        if tr.stats.npts == 0:
            print(f"Event {eid} has a trace with zero length, marking for removal")
            events_to_remove.append(eid)
            break

In [ ]:
if events_to_remove:
    print(f"Removing {len(events_to_remove)} events that do not have traces of the same length or have zero-length traces: {events_to_remove}")
    for eid in events_to_remove:
        del organized_waveforms[eid]
else:
    print("All events have traces of the same length and no zero-length traces found.")

In [ ]:
# Calculate quality control metrics for organized waveforms
print("Calculating quality control metrics for organized waveforms...")

# 1. Calculate S-wave SNR
organized_waveforms = calculate_snr_for_organized_waveforms(organized_waveforms)

# 2. Calculate geographic back-azimuth, for coordinate rotation later
organized_waveforms = calculate_back_azimuth_for_organized_waveforms(organized_waveforms, stations_df)

# 3. Calculate incidence angle
organized_waveforms = calculate_incidence_angle_eigenvalue_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

#4. Calculate P-wave rectilinearity
organized_waveforms = calculate_rectilinearity_jurkevics_for_organized_waveforms(organized_waveforms, p_arrival_variable='p_arrival_time', analysis_window=0.12)

In [ ]:
# Define passing_waveforms as those that meet all QC thresholds
passing_waveforms = apply_quality_control(organized_waveforms, QC_THRESHOLDS)

In [ ]:
# Save the passing_waveforms
metadata_df = save_passing_waveforms(passing_waveforms, output_dir='passing_waveforms_data_mldd_axec3')
display(metadata_df.head())

In [ ]:
# Reload the data
passing_waveforms_reloaded = load_passing_waveforms(output_dir='passing_waveforms_data_mldd_axec3')

# Verify the reload worked correctly
print(f"Reloaded events: {len(passing_waveforms_reloaded)}")
print(f"\nSample reloaded event (ID: {list(passing_waveforms_reloaded.keys())[0]}):")
sample_event = passing_waveforms_reloaded[list(passing_waveforms_reloaded.keys())[0]]
print(f"  Station: {sample_event['station']}")
print(f"  Origin time: {sample_event['origin_time']}")
print(f"  Number of traces: {len(sample_event['traces'])}")
print(f"  Back azimuth: {sample_event['back_azimuth']:.2f}°")

In [ ]:
passing_waveforms = passing_waveforms_reloaded.copy()

## 6. Shear-Wave Splitting Analysis

This section implements the core shear-wave splitting analysis using SWSPy with dynamic parameter estimation and comprehensive quality assessment.

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=2, last_window_start=1, 
                                                         first_window_end=1.8, last_window_end=2.2, n_win=10, s_pick_uncertainty=0.0395, mode='swspy', 
                                                         plot_results=False)
save_results_csv(results_swspy, file_name='splitting_results_swspy_axec3_apr_20_28_mldd_fixed_snr', mode='swspy')


In [ ]:
save_results_csv(results_swspy, file_name='splitting_results_swspy_axec3_apr_20_28_mldd', mode='swspy')

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using Baillard method...")
results_baillard = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=2, last_window_start=0, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0395, mode='baillard', 
                                                         plot_results=False)


In [ ]:
def save_results_csv(results, file_name, file_loc='/Users/mhemmett/Seismology/axial-splitting-ml/results/', mode='swspy'):
    # Create dataframe from results_swspy with event metadata and processing parameters

    # Initialize lists for each column
    data_rows = []

    for event_id, event_result in results.items():
        result = event_result['result']
        
        # Extract splitting measurements
        phi = result.get('phi')
        dt = result.get('dt')
        phi_error = result.get('phi_error')
        dt_error = result.get('dt_error')

        
        event_datetime = result.get('event_datetime')
        event_origin_time = result.get('event_origin_time')
        s_arrival_time = result.get('s_arrival_time')
        event_lat = result.get('event_lat')
        event_lon = result.get('event_lon')
        event_depth = result.get('event_depth')
        back_azimuth = result.get('back_azimuth')
        snr_horizontal = result.get('snr_horizontal')
        rectilinearity = result.get('rectilinearity_jurkevics')
        incidence = result.get('incidence_eigenvalue_jurkevics')

        if mode=='swspy':
            first_window_start = result.get('first_window_start')
            last_window_start = result.get('last_window_start')
            first_window_end = result.get('first_window_end')
            last_window_end = result.get('last_window_end')
            n_win = result.get('n_win')
            s_pick_uncertainty = result.get('s_pick_uncertainty')
            dominant_period = result.get('dominant_period')

            # Calculate numerical values for window start and end times
            first_window_start_pre_S_pick =  first_window_start * s_pick_uncertainty
            last_window_start_pre_S_pick = last_window_start * s_pick_uncertainty
            first_window_end_post_S_pick =  first_window_end * dominant_period
            last_window_end_post_S_pick = last_window_end * dominant_period

        # Create row dictionary
        row = {
            'event_id': event_id,
            'event_datetime': event_datetime,
            's_arrival_time' : s_arrival_time,
            'event_origin_time' : event_origin_time,
            'event_lat': event_lat,
            'event_lon': event_lon,
            'event_depth': event_depth,
            'back_azimuth': back_azimuth,
            'incidence': incidence,
            'snr_horizontal': snr_horizontal,
            'rectilinearity': rectilinearity,
            'phi': phi,
            'phi_error': phi_error,
            'dt': dt,
            'dt_error': dt_error,
        }
        
        data_rows.append(row)

        if mode == 'swspy':
            windowing_row = {
                # Processing parameters (constant values as specified)
                'dominant_period': dominant_period,
                'first_window_start_pre_S_pick': first_window_start_pre_S_pick,
                'last_window_start_pre_S_pick': last_window_start_pre_S_pick,
                'first_window_end_post_S_pick': first_window_end_post_S_pick,
                'last_window_end_post_S_pick': last_window_end_post_S_pick,
                'n_windows': n_win,
                'start_window_step_size': (first_window_start_pre_S_pick - last_window_start_pre_S_pick) / n_win,
                'end_window_step_size': (last_window_end_post_S_pick - first_window_end_post_S_pick) / n_win,
            }

            data_rows.append(windowing_row)

    # Create DataFrame
    results_df = pd.DataFrame(data_rows)

    # Display the dataframe
    print(f"\nSplitting Results DataFrame - {len(results_df)} events")
    print("=" * 100)
    display(results_df)

        # Save to CSV
    output_file = str(file_loc + file_name + '.csv')
    results_df.to_csv(output_file, index=False)
    print(f"\n✓ Saved to: {output_file}")

In [ ]:
save_results_csv(results_baillard, file_name='splitting_results_baillard_axec3_apr_20_28_mldd', mode='baillard')

In [ ]:
def plot_fast_direction_rose_eruption_comparison(results_dict, eruption_time=None, 
                                                  title_prefix="Fast Direction Distribution",
                                                  nbins=36, figsize=(16, 7), color='steelblue',
                                                  edgecolor='black', linewidth=0.5):
    """
    Create side-by-side 360° rose plots comparing fast directions before and after eruption.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    eruption_time : UTCDateTime
        Time of eruption onset (default: 2015-04-24T06:00:00)
    title_prefix : str
        Prefix for plot titles
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height) for combined plot
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    if eruption_time is None:
        eruption_time = UTCDateTime(2015, 4, 24, 6)  # Nooner and Chadwick 2016
    
    # Split results into before and after eruption
    results_before = {}
    results_after = {}
    
    for event_id, result in results_dict.items():
        event_time = UTCDateTime(result['result']['event_datetime'])
        if event_time < eruption_time:
            results_before[event_id] = result
        else:
            results_after[event_id] = result
    
    # Create figure with two subplots
    fig = plt.figure(figsize=figsize)
    
    # Before eruption plot (left)
    ax1 = fig.add_subplot(121, projection='polar')
    plot_rose_subplot(results_before, ax1, f"{title_prefix}\nBefore Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    # After eruption plot (right)
    ax2 = fig.add_subplot(122, projection='polar')
    plot_rose_subplot(results_after, ax2, f"{title_prefix}\nAfter Eruption", 
                      nbins, color, edgecolor, linewidth)
    
    plt.tight_layout()
    return fig, (ax1, ax2), (results_before, results_after)


def plot_rose_subplot(results_dict, ax, title, nbins, color, edgecolor, linewidth):
    """
    Helper function to plot rose diagram on a given axis.
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    if len(fast_directions) == 0:
        ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, 
                ha='center', va='center', fontsize=14)
        return
    
    fast_directions = np.array(fast_directions)
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean
    original_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']
        original_directions.append(np.deg2rad(phi))
    original_directions = np.array(original_directions)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)


def plot_fast_direction_rose(results_dict, title="Fast Direction Distribution", 
                              nbins=36, figsize=(8, 8), color='steelblue',
                              edgecolor='black', linewidth=0.5):
    """
    Create a 360° polar rose plot (histogram) of fast directions with 180° symmetry.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary of splitting results with event_id as keys
    title : str
        Title for the plot
    nbins : int
        Number of angular bins (default 36 = 10° bins for 360°)
    figsize : tuple
        Figure size (width, height)
    color : str
        Color for the histogram bars
    edgecolor : str
        Color for bar edges
    linewidth : float
        Width of bar edges
    """
    # Extract fast directions (phi) from results
    fast_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']  # in degrees (-90 to +90)
        # Convert to 0-360 range and add 180° symmetry
        # Map -90 to 90 range to 0 to 180, then add symmetric values
        phi_0_180 = phi + 90  # Convert to 0-180 range
        phi_rad_1 = np.deg2rad(phi_0_180)
        phi_rad_2 = np.deg2rad(phi_0_180 + 180)  # Add 180° symmetric value
        fast_directions.append(phi_rad_1)
        fast_directions.append(phi_rad_2)
    
    fast_directions = np.array(fast_directions)
    
    # Create polar histogram
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='polar')
    
    # Create histogram bins (0 to 2π for 0° to 360°)
    bins = np.linspace(0, 2*np.pi, nbins + 1)
    
    # Calculate histogram
    counts, bin_edges = np.histogram(fast_directions, bins=bins)
    
    # Calculate bin centers
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Width of each bar
    width = 2 * np.pi / nbins
    
    # Create the rose plot
    bars = ax.bar(bin_centers, counts, width=width, bottom=0.0,
                   color=color, edgecolor=edgecolor, linewidth=linewidth, alpha=0.7)
    
    # Set theta direction (clockwise from North)
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    
    # Set radial ticks
    ax.set_rlabel_position(45)
    
    # Add degree labels for full 360°
    tick_labels = ['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°']
    tick_positions = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels)
    
    # Adjust y-limit (counts are doubled due to symmetry)
    max_count = counts.max()
    ax.set_ylim(0, max_count * 1.2)
    
    # Add title with statistics
    n_measurements = len(fast_directions) // 2  # Divide by 2 since we doubled for symmetry
    # Calculate circular mean for original ±90° range
    original_directions = []
    for event_id, result in results_dict.items():
        phi = result['result']['phi']
        original_directions.append(np.deg2rad(phi))
    original_directions = np.array(original_directions)
    mean_direction = np.rad2deg(np.arctan2(np.sin(original_directions).sum(), 
                                           np.cos(original_directions).sum()))
    
    title_text = f"{title}\nN = {n_measurements}, Mean = {mean_direction:.1f}°"
    ax.set_title(title_text, va='bottom', fontsize=12, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    return fig, ax

In [ ]:

def output_plots(filename, first_start, last_start, first_end, last_end, station):
    results_df_axec2_2_0_1_5_2_5 = pd.read_csv(filename)

    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

    # Convert results_df_axec2 event_datetime to datetime for plotting
    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

    fig, (ax1, ax2), (results_before, results_after) = plot_fast_direction_rose_eruption_comparison(
        results_df_axec2_2_0_1_5_2_5,
        title_prefix="Fast Direction Distribution, " + str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        nbins=36,  # 10° bins
        color='steelblue',
        figsize=(16, 7)
    )
    plt.show()

    results_df_axec2_2_0_1_5_2_5 = pd.read_csv(filename)

    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: UTCDateTime(x))

    # Filter results_df_axec2 to be +/- 24 hrs of the eruption onset
    eruption_time = UTCDateTime(2015, 4, 24, 6)
    time_window = 24 * 3600  # 24 hours in seconds
    start_time = eruption_time - time_window
    end_time = eruption_time + time_window
    mask = (results_df_axec2_2_0_1_5_2_5['event_datetime'] >= start_time) & (results_df_axec2_2_0_1_5_2_5['event_datetime'] <= end_time)
    results_df_axec2_2_0_1_5_2_5 = results_df_axec2_2_0_1_5_2_5[mask]

    # Convert results_df_axec2 event_datetime to datetime for plotting
    results_df_axec2_2_0_1_5_2_5['event_datetime'] = results_df_axec2_2_0_1_5_2_5['event_datetime'].apply(lambda x: x.datetime)

    results_df_axec2_baillard_phi = results_df_axec2_2_0_1_5_2_5.copy()
    # Conversion formula: phi_ccw_from_E = 90° - phi_cw_from_N
    results_df_axec2_baillard_phi['phi'] = 90 - results_df_axec2_baillard_phi['phi']

    # Handle wrapping: keep values in -90° to +90° range
    # If result > 90°, subtract 180°
    # If result < -90°, add 180°
    results_df_axec2_baillard_phi['phi'] = results_df_axec2_baillard_phi['phi'].apply(
        lambda x: x - 180 if x > 90 else (x + 180 if x < -90 else x)
    )

    # Now plot on AXEC2 data

    fig, ax = plot_dt_timeseries_movehisto(
        results_df_axec2_baillard_phi, 
        station=str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        x_width=100,  # 1 day time window (shorter)
        x_overlap=0.9,
        y_width=1,          # 1 sample bins (finer)
        y_overlap=0.9,
        flag_smooth=True,
        x_filter_per=5,     # Less smoothing in time (5%)
        y_filter_per=5,      # Less smoothing in dt (5%)
        figsize=(6,8)
    )
    plt.show()

    fig, ax = plot_phi_timeseries_movehisto(
        results_df_axec2_baillard_phi, 
        station=str(station) + ", SWSPy: " + str(first_start) + "-"+ str(last_start) + "σ, " + str(first_end) + "-" + str(last_end) + " Tmid End",
        x_width=100,  # 1 day time window (shorter)
        x_overlap=0.9,
        y_width=180/40,          # 5 degree bins (finer)
        y_overlap=0.9,
        flag_smooth=True,
        x_filter_per=5,     # Less smoothing in time (5%)
        y_filter_per=5,      # Less smoothing in phi (5%)
        figsize=(6,8)
    )
    plt.show()

    return results_df_axec2_2_0_1_5_2_5


In [ ]:
output_plots(filename='../results/splitting_results_swspy_axec3_apr_20_28_mldd.csv', first_start='2', last_start='1', first_end='1.8', last_end='2.2')

In [ ]:
output_plots(filename='../results/splitting_results_baillard_axec3_apr_20_28_mldd.csv', first_start='0..02s', last_start='0', first_end='2', last_end='2')

In [ ]:
# Shear-wave splitting analysis using dynamic splitting parameters
print("Performing shear-wave splitting analysis using SWSPy method...")
results_swspy_1_5_2_5 = perform_splitting_on_organized_waveforms(passing_waveforms, first_window_start=2, last_window_start=1, 
                                                         first_window_end=1.5, last_window_end=2.5, n_win=10, s_pick_uncertainty=0.0395, mode='swspy', 
                                                         plot_results=False)
save_results_csv(results_swspy_1_5_2_5, file_name='splitting_results_swspy_axec3_apr_20_28_mldd_1_5_2_5', mode='swspy')